In [ ]:
DEEPFAKE AUDIO DETECTION 

In [ ]:
import torch
import torch.nn as nn
import librosa
import numpy as np
from torch.utils.data import DataLoader
from audio_dataset import AudioDataset
from model import DeepfakeCNN


In [ ]:
train_dataset = AudioDataset("dataset/train")
val_dataset = AudioDataset("dataset/validation")
test_dataset = AudioDataset("dataset/test")

In [ ]:
import torch
import torch.nn as nn


class DeepfakeCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # Input: (1, 128, 100)

            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(64 * 16 * 12, 128),
            nn.ReLU(),

            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from audio_dataset import AudioDataset
from model import DeepfakeCNN

# Create dataset
train_dataset = AudioDataset("dataset/train")

# Create dataloader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

# Create model
model = DeepfakeCNN()

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# Number of epochs
epochs = 5

for epoch in range(epochs):

    running_loss = 0.0

    for spectrograms, labels in train_loader:

        # Add channel dimension
        spectrograms = spectrograms.unsqueeze(1)

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(spectrograms)

        # Compute loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {average_loss:.4f}"
    )
    torch.save(model.state_dict(), "model.pth")

    print("Model saved successfully!")

In [ ]:
import torch
from torch.utils.data import DataLoader

from audio_dataset import AudioDataset
from model import DeepfakeCNN

# Load validation dataset
val_dataset = AudioDataset("dataset/val")

# Create dataloader
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

# Create model
model = DeepfakeCNN()

# Load trained weights
model.load_state_dict(torch.load("model.pth"))

# Evaluation mode
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for spectrograms, labels in val_loader:

        spectrograms = spectrograms.unsqueeze(1)

        outputs = model(spectrograms)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Validation Accuracy:", accuracy, "%")

In [ ]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve
)
from scipy.optimize import brentq
from scipy.interpolate import interp1d

from audio_dataset import AudioDataset
from model import DeepfakeCNN


# -------------------------------
# Load Test Dataset
# -------------------------------
test_dataset = AudioDataset("dataset/test")

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)


# -------------------------------
# Load Trained Model
# -------------------------------
model = DeepfakeCNN()

model.load_state_dict(
    torch.load(
        "model.pth",
        map_location=torch.device("cpu")
    )
)

model.eval()


# -------------------------------
# Testing Variables
# -------------------------------
correct = 0
total = 0

all_labels = []
all_predictions = []
all_scores = []


# -------------------------------
# Testing Loop
# -------------------------------
with torch.no_grad():

    for spectrograms, labels in test_loader:

        # Add channel dimension
        spectrograms = spectrograms.unsqueeze(1)

        # Forward pass
        outputs = model(spectrograms)

        # Predicted class
        _, predicted = torch.max(outputs, 1)

        # Accuracy calculation
        total += labels.size(0)

        correct += (predicted == labels).sum().item()

        # Store predictions
        all_labels.extend(labels.numpy())

        all_predictions.extend(predicted.numpy())

        # Store probabilities for EER
        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        real_scores = probabilities[:, 1]

        all_scores.extend(real_scores.numpy())


# -------------------------------
# Overall Accuracy
# -------------------------------
accuracy = 100 * correct / total

print(
    f"\nTest Accuracy: {accuracy:.2f}%"
)


# -------------------------------
# Confusion Matrix
# -------------------------------
print("\nConfusion Matrix:")

print(
    confusion_matrix(
        all_labels,
        all_predictions
    )
)


# -------------------------------
# Classification Report
# -------------------------------
print("\nClassification Report:")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["Fake", "Real"]
    )
)


# -------------------------------
# Equal Error Rate (EER)
# -------------------------------
fpr, tpr, thresholds = roc_curve(
    all_labels,
    all_scores
)

eer = brentq(
    lambda x: 1.0 - x - interp1d(fpr, tpr)(x),
    0.0,
    1.0
)

print(
    f"EER: {eer * 100:.2f}%"
)


In [ ]:
## Final Results

Accuracy: 99.90%

EER: 0.26%

Confusion Matrix:
[[1600    0]
 [   2  378]]